# 00 — Setup de la sesión de ColabCorré esto primero en cada sesión. Colab arranca de cero cada vez: monta Drive,clona el repo, instala las herramientas y verifica que arranquen de verdad.Es el equivalente de `check_env.sh`, recortado a lo que Colab necesita —acá solose descarga y se valida, no se alinea— así que no hacen falta bowtie, fastp niViennaRNA.

## Preámbulo: montar Drive y clonar el repoEl repo es público, así que el clon no necesita credenciales. **Los notebooksllaman a los scripts del repo en vez de reimplementarlos**: el criterio deselección de corridas y el de verificación de ensamblados tienen que vivir enun solo lugar, o dejan de ser reproducibles.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import os, pathlibDRIVE = pathlib.Path('/content/drive/MyDrive/tesis')CLON  = pathlib.Path('/content/tesis')assert DRIVE.exists(), f'no veo {DRIVE} — ¿montaste la cuenta correcta?'print('Drive OK:', DRIVE)

In [ ]:
import subprocessif CLON.exists():    print(subprocess.run(['git','-C',str(CLON),'pull','--ff-only'],                         capture_output=True, text=True).stdout)else:    print(subprocess.run(['git','clone','--depth','1','https://github.com/youkonskernel-afk/tesis.git',str(CLON)],                         capture_output=True, text=True).stderr)print(subprocess.run(['git','-C',str(CLON),'log','--oneline','-1'],                     capture_output=True, text=True).stdout)

## Herramientas`sra-tools` trae `prefetch` y `vdb-validate`. Se baja el tarball oficial de NCBIen vez de usar `apt`, porque el paquete de Ubuntu suele ir varias versionesatrás. Si la URL cambia, ajustá `SRA_VER`.El entorno `srna2` del proyecto pinea sra-tools 3.4.1. Acá la versión puedediferir y no es grave: Colab solo descarga y valida, no produce resultados queentren en la tesis. Lo que sí importa es que `vdb-validate` exista.

In [ ]:
SRA_VER = '3.1.1'url = f'https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/{SRA_VER}/sratoolkit.{SRA_VER}-ubuntu64.tar.gz'!apt-get -qq install -y jq > /dev/null 2>&1!wget -q -O /tmp/sra.tar.gz "$url" && tar -xzf /tmp/sra.tar.gz -C /optimport glob, oscands = glob.glob(f'/opt/sratoolkit.{SRA_VER}*/bin')if cands:    os.environ['PATH'] = cands[0] + ':' + os.environ['PATH']    print('sra-tools en', cands[0])else:    print('AVISO: no se desempaquetó sra-tools; probá con apt-get install sra-toolkit')

## Verificar que arrancanNo alcanza con que el binario exista: tiene que ejecutar. Es la misma lógica que`check_env.sh` aplica en la máquina local.

In [ ]:
import shutil, subprocessdef prueba(cmd, args=['--version']):    ruta = shutil.which(cmd)    if not ruta:        return f'[MAL] {cmd}: no está en PATH'    try:        r = subprocess.run([cmd] + args, capture_output=True, text=True, timeout=60)        v = (r.stdout + r.stderr).strip().split('\n')[0]        return f'[OK ] {cmd}: {v}'    except Exception as e:        return f'[MAL] {cmd}: no arranca ({e})'for c in ['prefetch', 'vdb-validate', 'curl', 'jq', 'git']:    print(prueba(c))

## Árbol de Drive

In [ ]:
esperadas = ['00_manifiestos','10_bam','20_yasma','30_qc','40_features',             '50_modelos','60_figuras','70_genomas','80_sra']for d in esperadas:    p = DRIVE / d    print(f"[{'OK ' if p.exists() else 'FALTA'}] {d}")import shutil as _shlibre = _sh.disk_usage('/content').free / 1e9print(f'\ndisco efímero de la VM: {libre:.0f} GB libres')

## Después de esto- `descarga_genomas.ipynb` — verificar y bajar los ensamblados- `10_descarga_runs.ipynb` — resolver el manifiesto y bajar los `.sra`- `90_estado.ipynb` — ver qué falta